In [ ]:
#  CHUNK 1:

from datasets import load_dataset, Audio
import re
import json

print("Loading dataset...")
ds = load_dataset("amaai-lab/DisfluencySpeech", split="train")

# wav2vec2 only works with 16kHz  this line handles the conversion
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

def parse_transcript(text):
    """
    walks through the annotated transcript token by token and returns:
      clean_words        — list of all spoken words with tags stripped
      disfluent_indices  — list of word indices that are disfluencies
    e.g. "I {F uh} went" → words=["I","uh","went"], disfluent=[1]
    """
    clean_words = []
    disfluent_indices = []

    # split on whitespace but keep tag groups like {F uh} together
    tokens = re.findall(r'\{[^}]+\}|\S+', text)

    for token in tokens:
        # filled pause {F ...} or discourse marker {D ...}
        m = re.match(r'\{[FD](.*?)\}', token)
        if m:
            inner = m.group(1).strip()
            for word in inner.split():
                disfluent_indices.append(len(clean_words))
                clean_words.append(word)
        else:
            # normal word — strip any leftover brackets from repetition tags
            word = re.sub(r'[\[\]+]', '', token).strip()
            if word:
                clean_words.append(word)

    return clean_words, disfluent_indices

def clean_disfluency_tags(batch):
    text = batch["transcript_annotated"]

    clean_words, disfluent_indices = parse_transcript(text)

    return {
        "text_cleaned"      : " ".join(clean_words),
        "total_words"       : len(clean_words),
        # store as json string  HuggingFace datasets can't hold a list of ints directly
        "disfluent_indices" : json.dumps(disfluent_indices),
        # clip-level label still useful as a quick filter
        "has_disfluency"    : 1 if len(disfluent_indices) > 0 else 0,
    }

print("Preprocessing text...")
processed_ds = ds.map(clean_disfluency_tags)

# quick check  make sure the cleaning actually worked
example = processed_ds[0]
print(f"Original          : {example['transcript_annotated']}")
print(f"Cleaned           : {example['text_cleaned']}")
print(f"Total words       : {example['total_words']}")
print(f"Disfluent at idx  : {example['disfluent_indices']}  ← word positions")
print(f"Clip label        : {example['has_disfluency']}  (1 = bad speech, 0 = clean)")
print(f"Audio             : {example['audio']['array'].shape} @ {example['audio']['sampling_rate']}Hz")


In [ ]:
#  CHUNK 4: Feature Extraction


import json, gc, os
import numpy as np
from transformers import AutoFeatureExtractor
from datasets import Dataset, DatasetDict, concatenate_datasets

SR                         = 16000
CHUNK_SIZE                 = SR * 5   # 5 seconds of samples
MIN_DISFLUENCIES_TO_BE_BAD = 2        # 1 filler = clean,  2+ = bad speech
BATCH_SIZE                 = 400
SAVE_DIR                   = "/content/chunks_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Loading train and test splits from the dataset...")
ds_train = load_dataset("amaai-lab/DisfluencySpeech", split="train")
ds_test  = load_dataset("amaai-lab/DisfluencySpeech", split="test")

ds_train = ds_train.cast_column("audio", Audio(sampling_rate=SR))
ds_test  = ds_test.cast_column("audio",  Audio(sampling_rate=SR))

processed_train = ds_train.map(clean_disfluency_tags)
processed_test  = ds_test.map(clean_disfluency_tags)

print(f"train clips: {len(processed_train)}")
print(f"test clips : {len(processed_test)}")

model_checkpoint = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_checkpoint)

def get_filler_timeline(t_start, t_end, disfluent_indices, clean_words, total_words, total_duration):
    timeline = []
    if total_words == 0 or total_duration == 0:
        return timeline
    for w_idx in disfluent_indices:
        word_time = (w_idx / total_words) * total_duration
        if t_start <= word_time < t_end:
            timeline.append({
                "word"          : clean_words[w_idx] if w_idx < len(clean_words) else "?",
                "time_in_chunk" : round(word_time - t_start, 3),
                "time_in_clip"  : round(word_time, 3),
            })
    return timeline

def chop_and_label(sample):
    audio       = np.array(sample["audio"]["array"], dtype=np.float32)
    total_dur   = len(audio) / SR
    d_indices   = json.loads(sample["disfluent_indices"])
    total_words = sample["total_words"]
    clean_words = sample["text_cleaned"].split()

    chunks_iv, chunks_am, chunks_lbl, chunks_tl = [], [], [], []

    for start in range(0, len(audio), CHUNK_SIZE):
        chunk = audio[start : start + CHUNK_SIZE]

        if len(chunk) < CHUNK_SIZE:
            chunk = np.pad(chunk, (0, CHUNK_SIZE - len(chunk)))

        feats = feature_extractor(
            [chunk],
            sampling_rate=SR,
            max_length=CHUNK_SIZE,
            truncation=True,
            padding="max_length",
            return_tensors="np"
        )

        t_start  = start / SR
        t_end    = t_start + 5.0
        timeline = get_filler_timeline(t_start, t_end, d_indices, clean_words, total_words, total_dur)
        lbl      = 1 if len(timeline) >= MIN_DISFLUENCIES_TO_BE_BAD else 0

        chunks_iv.append(feats["input_values"][0].astype(np.float16))
        if "attention_mask" in feats:
            chunks_am.append(feats["attention_mask"][0])
        chunks_lbl.append(lbl)
        chunks_tl.append(json.dumps(timeline))

    del audio
    return chunks_iv, chunks_am, chunks_lbl, chunks_tl

def process_split(split_ds, split_name):
    batch_paths = []
    total_clips = len(split_ds)

    for batch_start in range(0, total_clips, BATCH_SIZE):
        batch_end  = min(batch_start + BATCH_SIZE, total_clips)
        batch_num  = batch_start // BATCH_SIZE
        print(f"  {split_name} — batch {batch_num+1}  "
              f"(clips {batch_start}–{batch_end} of {total_clips})...")

        b_iv, b_am, b_lbl, b_tl = [], [], [], []

        for i in range(batch_start, batch_end):
            ivs, ams, lbls, tls = chop_and_label(split_ds[i])
            b_iv.extend(ivs)
            b_am.extend(ams)
            b_lbl.extend(lbls)
            b_tl.extend(tls)

        data_dict = {
            "input_values"   : b_iv,
            "labels"         : b_lbl,
            "filler_timeline": b_tl,
        }
        if b_am:
            data_dict["attention_mask"] = b_am

        batch_ds   = Dataset.from_dict(data_dict)
        save_path  = os.path.join(SAVE_DIR, f"{split_name}_batch_{batch_num:04d}")
        batch_ds.save_to_disk(save_path)
        batch_paths.append(save_path)

        del b_iv, b_am, b_lbl, b_tl, batch_ds, data_dict
        gc.collect()

    print(f"  combining {len(batch_paths)} batches for {split_name}...")
    from datasets import load_from_disk
    all_batches = [load_from_disk(p) for p in batch_paths]
    full_ds     = concatenate_datasets(all_batches)

    labels      = full_ds["labels"]
    bad_count   = sum(labels)
    clean_count = len(labels) - bad_count
    print(f"  {split_name} done — {len(split_ds)} clips → {len(full_ds)} chunks")
    print(f"    bad speech : {bad_count}")
    print(f"    clean      : {clean_count}")

    for tl in full_ds["filler_timeline"][:30]:
        parsed = json.loads(tl)
        if parsed:
            print(f"\n  sample filler timeline:")
            for entry in parsed:
                print(f"    '{entry['word']}' at {entry['time_in_chunk']}s into chunk "
                      f"(= {entry['time_in_clip']}s in full clip)")
            break

    return full_ds

print("\nStarting feature extraction...")
train_encoded = process_split(processed_train, "train")
test_encoded  = process_split(processed_test,  "test")

split_ds = DatasetDict({"train": train_encoded, "test": test_encoded})

print(f"\ntrain: {len(split_ds['train'])} chunks")
print(f"test : {len(split_ds['test'])} chunks")


In [ ]:
#  CHUNK 5: Model

import os
os.system("pip install -q accelerate evaluate scikit-learn")

import torch
import torch.nn as nn
import numpy as np
import evaluate
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer
from sklearn.utils.class_weight import compute_class_weight

#  1. custom collator
def data_collator(features):
    batch = {}
    batch["input_values"] = torch.tensor(
        np.array([f["input_values"] for f in features]), dtype=torch.float32
    )
    if "attention_mask" in features[0]:
        batch["attention_mask"] = torch.tensor(
            np.array([f["attention_mask"] for f in features]), dtype=torch.long
        )
    batch["labels"] = torch.tensor(
        [f["labels"] for f in features], dtype=torch.long
    )
    return batch

#  2. metrics accuracy + F1 so we can see per-class performance
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    refs        = eval_pred.label_ids
    acc = accuracy_metric.compute(predictions=predictions, references=refs)
    f1  = f1_metric.compute(predictions=predictions, references=refs, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

#  3. compute class weights from training labels
#    sklearn returns weights ordered by sorted class index → [w_clean, w_bad]
train_labels = np.array(split_ds["train"]["labels"])
classes      = np.array([0, 1])
weights      = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)
print(f"Class weights — Clean: {weights[0]:.3f}  Bad Speech: {weights[1]:.3f}")

#  4. custom Trainer that uses weighted cross-entropy
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # move weights to the same device as the model
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        device = logits.device
        w = self.class_weights.to(device) if self.class_weights is not None else None
        loss_fn = nn.CrossEntropyLoss(weight=w)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

#  5. load model and FREEZE the CNN feature extractor
print("Loading model...")
model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,
    label2id={"Clean": 0, "Bad Speech": 1},
    id2label={0: "Clean", 1: "Bad Speech"}
)

model.wav2vec2.feature_extractor._freeze_parameters()
frozen = sum(p.numel() for p in model.wav2vec2.feature_extractor.parameters())
total  = sum(p.numel() for p in model.parameters())
print(f"Frozen {frozen:,} / {total:,} params ({100*frozen/total:.1f}%)")
print("Only transformer layers + classifier head will be updated.")


In [ ]:
#  CHUNK 6: Training

training_args = TrainingArguments(
    output_dir                  = "./bad_speech_model",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = 1e-5,          #  lowered (CNN frozen)
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,             #  effective batch = 16
    num_train_epochs            = 5,             #  was 2
    warmup_ratio                = 0.05,          #  shorter warmup
    weight_decay                = 0.01,          #  L2 regularisation
    lr_scheduler_type           = "cosine",      #  smooth LR decay
    logging_steps               = 10,
    report_to                   = "none",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",    #  was accuracy
    greater_is_better           = True,
    remove_unused_columns       = False,
    group_by_length             = False,
)

trainer = WeightedTrainer(
    model         = model,
    args          = training_args,
    train_dataset = split_ds["train"],
    eval_dataset  = split_ds["test"],
    compute_metrics = compute_metrics,
    data_collator = data_collator,
    class_weights = class_weights_tensor,   #  weighted loss
)

print("Training...")
trainer.train()

final_stats = trainer.evaluate()
print(f"Final accuracy : {final_stats['eval_accuracy']:.2%}")
print(f"Final F1-macro : {final_stats['eval_f1_macro']:.4f}")


In [ ]:
# CHUNK 7 full evaluation

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("Running predictions on the test set...")

pred_out = trainer.predict(split_ds["test"])

logits = np.array(pred_out.predictions, dtype=np.float32)

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

preds = np.argmax(probs, axis=1)

actuals = np.array(pred_out.label_ids, dtype=np.int64)

# accuracy
accuracy = float((preds == actuals).mean())

print("\n" + "="*50)
print(f"Test Accuracy : {accuracy:.2%}")
print("="*50)

bad_probs = probs[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(actuals, preds)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Clean", "Bad Speech"]
)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")

axes[1].hist(bad_probs[actuals == 0], bins=30, alpha=0.7, label="Actual Clean")
axes[1].hist(bad_probs[actuals == 1], bins=30, alpha=0.7, label="Actual Bad Speech")
axes[1].set_xlabel("Predicted probability of Bad Speech")
axes[1].set_ylabel("Number of samples")
axes[1].set_title("Probability Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("fillerword_eval.png", dpi=120)
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(actuals, preds, target_names=["Clean", "Bad Speech"]))

# Display a few predictions
print("\nSample predictions:")
print(f"{'#':<4} {'Actual':<12} {'Predicted':<12} {'Bad_Prob':>10}")
print("-" * 45)

for i, (a, p, bp) in enumerate(zip(actuals[:10], preds[:10], bad_probs[:10])):
    actual_label = "Bad Speech" if a == 1 else "Clean"
    pred_label = "Bad Speech" if p == 1 else "Clean"
    print(f"{i+1:<4} {actual_label:<12} {pred_label:<12} {bp:10.3f}")

In [ ]:
#  CHUNK 8: Save

print("Saving model...")
trainer.save_model("./bad_speech_model_final")
feature_extractor.save_pretrained("./bad_speech_model_final")
print("Saved to ./bad_speech_model_final")


In [ ]:
#  CHUNK 9: Test on PodcastFillers (real world audio, no labels)

import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

SR         = 16000
CHUNK_SIZE = SR * 5
N_CLIPS    = 20

print("Loading model from disk...")
model_path = "./bad_speech_model_final"
fe         = AutoFeatureExtractor.from_pretrained(model_path)
model      = AutoModelForAudioClassification.from_pretrained(model_path)
model.eval()

print("Streaming 20 clips from PodcastFillers...")
podcast_ds = load_dataset("bookbot/PodcastFillers-filtered", streaming=True)

try:
    stream = iter(podcast_ds["train"])
except KeyError:
    stream = iter(podcast_ds)

def predict_clip(audio_array, sr):
    if sr != SR:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SR)

    audio         = np.array(audio_array, dtype=np.float32)
    clip_dur      = len(audio) / SR
    chunk_results = []

    for start in range(0, len(audio), CHUNK_SIZE):
        chunk = audio[start : start + CHUNK_SIZE]
        if len(chunk) < CHUNK_SIZE:
            chunk = np.pad(chunk, (0, CHUNK_SIZE - len(chunk)))

        feats = fe(
            [chunk],
            sampling_rate=SR,
            max_length=CHUNK_SIZE,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        with torch.no_grad():
            logits = model(**feats).logits

        probs      = torch.softmax(logits, dim=-1)[0]
        pred_idx   = torch.argmax(probs).item()
        label      = model.config.id2label[pred_idx]
        confidence = probs[pred_idx].item()

        t_start = start / SR
        t_end   = min(t_start + 5.0, clip_dur)
        chunk_results.append({
            "window"    : f"{t_start:.1f}s – {t_end:.1f}s",
            "label"     : label,
            "confidence": round(confidence, 3),
        })

    any_bad      = any(r["label"] == "Bad Speech" for r in chunk_results)
    clip_verdict = "Bad Speech" if any_bad else "Clean"
    return chunk_results, clip_verdict, round(clip_dur, 2)

print(f"\nRunning inference on {N_CLIPS} clips...\n")
print("=" * 60)

for clip_num in range(N_CLIPS):
    try:
        sample = next(stream)
    except StopIteration:
        print("dataset ran out of clips early")
        break

    audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
    sr          = sample["audio"]["sampling_rate"]

    chunk_results, verdict, dur = predict_clip(audio_array, sr)

    print(f"Clip {clip_num+1:>2}  ({dur}s)  →  overall: {verdict}")
    for r in chunk_results:
        flag = "⚠" if r["label"] == "Bad Speech" else " "
        print(f"   {flag} {r['window']:<18} {r['label']:<12}  confidence: {r['confidence']:.1%}")
    print()

print("=" * 60)
print("Done.")
